# GRDP per Capita — Silver
2021–2025 | Whole pesos per person | Initial load only.


## 1. Source coverage
Review the available years and source price labels before creating the table.


In [0]:
SELECT
  TRY_CAST(year AS INT) AS year,
  TRIM(price_basis) AS price_basis,
  COUNT(*) AS total_rows
FROM pondolens.dev_bronze.grdp_per_capita
WHERE TRY_CAST(year AS INT) BETWEEN 2021 AND 2025
GROUP BY TRY_CAST(year AS INT), TRIM(price_basis)
ORDER BY year, price_basis;

## 2. Create table and clean
Trim codes, remove extra whitespace, map Philippines to PHL and MIMAROPA to IV-B, and classify Philippines as national. Current-price basis becomes the observation year; constant-price basis becomes 2018. Initial IDs start at 100000001.


In [0]:
CREATE TABLE IF NOT EXISTS pondolens.dev_silver.grdp_per_capita
USING DELTA
AS
WITH normalized AS (
  SELECT
    NULLIF(UPPER(TRIM(region_id)), '') AS source_region_code,
    NULLIF(
      TRIM(REGEXP_REPLACE(region, '\\s+', ' ')), ''
    ) AS region_name,
    TRY_CAST(year AS INT) AS year,

    TRY_CAST(
      ROUND(
        TRY_CAST(GRDP_per_capita_amount AS DECIMAL(30, 15)), 0
      ) AS BIGINT
    ) AS grdp_per_capita,

    LOWER(TRIM(price_basis)) AS source_price_basis,
    NULLIF(LOWER(TRIM(geographic_level)), '')
      AS source_geographic_level

  FROM pondolens.dev_bronze.grdp_per_capita
  WHERE TRY_CAST(year AS INT) BETWEEN 2021 AND 2025
),
cleaned AS (
  SELECT
    CASE
      WHEN UPPER(region_name) = 'PHILIPPINES' THEN 'PHL'
      WHEN UPPER(region_name) = 'MIMAROPA REGION' THEN 'IV-B'
      ELSE source_region_code
    END AS region_code,
    region_name,
    year,
    grdp_per_capita,
    CASE
      WHEN source_price_basis = 'at current prices' THEN year
      WHEN source_price_basis = 'at constant 2018 prices' THEN 2018
      ELSE NULL
    END AS price_basis,
    CASE
      WHEN UPPER(region_name) = 'PHILIPPINES' THEN 'national'
      ELSE source_geographic_level
    END AS geographic_level
  FROM normalized
)
SELECT
  CAST(
    100000000 + ROW_NUMBER() OVER (
      ORDER BY year, region_name, price_basis
    ) AS INT
  ) AS grdp_per_capita_id,
  region_code,
  region_name,
  year,
  grdp_per_capita,
  price_basis,
  geographic_level,
  CONCAT(
    'silver_grdp_pc_',
    DATE_FORMAT(CURRENT_TIMESTAMP(), 'yyyyMMdd_HHmmss')
  ) AS processing_run_id,
  CURRENT_TIMESTAMP() AS processed_at
FROM cleaned;

### 3. Review data and types
Confirm the saved columns and numeric data types.

In [0]:
SELECT *
FROM pondolens.dev_silver.grdp_per_capita
ORDER BY grdp_per_capita_id;

DESCRIBE TABLE pondolens.dev_silver.grdp_per_capita;

### 4. Reconcile row counts
Expected for the checked source: 185 Bronze rows and 185 Silver rows, with MATCH status.

In [0]:
WITH counts AS (
  SELECT
    (SELECT COUNT(*) FROM pondolens.dev_bronze.grdp_per_capita
     WHERE TRY_CAST(year AS INT) BETWEEN 2021 AND 2025) AS bronze_rows,
    (SELECT COUNT(*) FROM pondolens.dev_silver.grdp_per_capita) AS silver_rows
)
SELECT *,
  CASE WHEN bronze_rows = silver_rows THEN 'MATCH' ELSE 'MISMATCH' END AS status
FROM counts;

### 5. Check required values and valid ranges
All null counts should be zero. The invalid-row query should return no rows.

In [0]:
SELECT
  COUNT(*) AS total_rows,
  COUNT_IF(grdp_per_capita_id IS NULL) AS null_grdp_per_capita_id,
  COUNT_IF(region_code IS NULL) AS null_region_code,
  COUNT_IF(region_name IS NULL) AS null_region_name,
  COUNT_IF(year IS NULL) AS null_year,
  COUNT_IF(grdp_per_capita IS NULL) AS null_grdp_per_capita,
  COUNT_IF(price_basis IS NULL) AS null_price_basis,
  COUNT_IF(geographic_level IS NULL) AS null_geographic_level,
  COUNT_IF(processing_run_id IS NULL) AS null_processing_run_id,
  COUNT_IF(processed_at IS NULL) AS null_processed_at
FROM pondolens.dev_silver.grdp_per_capita;

In [0]:
SELECT *
FROM pondolens.dev_silver.grdp_per_capita
WHERE grdp_per_capita_id IS NULL OR grdp_per_capita_id <= 100000000
   OR region_code IS NULL OR TRIM(region_code) = ''
   OR region_name IS NULL OR TRIM(region_name) = ''
   OR year IS NULL OR year NOT BETWEEN 2021 AND 2025
   OR grdp_per_capita IS NULL OR grdp_per_capita <= 0
   OR price_basis IS NULL OR (price_basis <> 2018 AND price_basis <> year)
   OR geographic_level IS NULL OR geographic_level NOT IN ('region', 'national')
   OR (geographic_level = 'national' AND
       (region_code <> 'PHL' OR UPPER(region_name) <> 'PHILIPPINES'))
   OR (geographic_level = 'region' AND
       (region_code = 'PHL' OR UPPER(region_name) = 'PHILIPPINES'))
   OR processing_run_id IS NULL OR TRIM(processing_run_id) = ''
   OR processed_at IS NULL;

### 6. Check duplicate keys
Both queries should return no rows. A unique generated ID does not prove that business records are unique.

In [0]:
SELECT grdp_per_capita_id, COUNT(*) AS row_count
FROM pondolens.dev_silver.grdp_per_capita
GROUP BY grdp_per_capita_id
HAVING COUNT(*) > 1;

SELECT year, region_code, price_basis, geographic_level, COUNT(*) AS row_count
FROM pondolens.dev_silver.grdp_per_capita
GROUP BY year, region_code, price_basis, geographic_level
HAVING COUNT(*) > 1;

### 7. Compare transformed source records
Expected: no rows. This compares Bronze against Silver after the same cleaning and whole-peso rounding; it does not independently verify the original Excel. EXCEPT ALL also detects extra or missing copies.


In [0]:
WITH expected AS (
WITH normalized AS (
  SELECT
    NULLIF(UPPER(TRIM(region_id)), '') AS source_region_code,
    NULLIF(
      TRIM(REGEXP_REPLACE(region, '\\s+', ' ')), ''
    ) AS region_name,
    TRY_CAST(year AS INT) AS year,

    TRY_CAST(
      ROUND(
        TRY_CAST(GRDP_per_capita_amount AS DECIMAL(30, 15)), 0
      ) AS BIGINT
    ) AS grdp_per_capita,

    LOWER(TRIM(price_basis)) AS source_price_basis,
    NULLIF(LOWER(TRIM(geographic_level)), '')
      AS source_geographic_level

  FROM pondolens.dev_bronze.grdp_per_capita
  WHERE TRY_CAST(year AS INT) BETWEEN 2021 AND 2025
),
cleaned AS (
  SELECT
    CASE
      WHEN UPPER(region_name) = 'PHILIPPINES' THEN 'PHL'
      WHEN UPPER(region_name) = 'MIMAROPA REGION' THEN 'IV-B'
      ELSE source_region_code
    END AS region_code,
    region_name,
    year,
    grdp_per_capita,
    CASE
      WHEN source_price_basis = 'at current prices' THEN year
      WHEN source_price_basis = 'at constant 2018 prices' THEN 2018
      ELSE NULL
    END AS price_basis,
    CASE
      WHEN UPPER(region_name) = 'PHILIPPINES' THEN 'national'
      ELSE source_geographic_level
    END AS geographic_level
  FROM normalized
)

SELECT region_code, region_name, year, grdp_per_capita, price_basis, geographic_level
FROM cleaned
), actual AS (
  SELECT region_code, region_name, year, grdp_per_capita, price_basis, geographic_level
  FROM pondolens.dev_silver.grdp_per_capita
), missing AS (
  SELECT * FROM expected EXCEPT ALL SELECT * FROM actual
), extra AS (
  SELECT * FROM actual EXCEPT ALL SELECT * FROM expected
)
SELECT 'MISSING_OR_CHANGED' AS issue, * FROM missing
UNION ALL
SELECT 'EXTRA_OR_CHANGED' AS issue, * FROM extra;

### 8. Review geographic coverage
Expected per year: 18 regional records for each price basis and one national current-price record. The source has no national constant-price record.


In [0]:
SELECT year, geographic_level, price_basis,
  COUNT(*) AS total_rows,
  COUNT(DISTINCT region_code) AS distinct_regions
FROM pondolens.dev_silver.grdp_per_capita
GROUP BY year, geographic_level, price_basis
ORDER BY year, geographic_level, price_basis;